# Data preparation (provenance only)

**This notebook is intentionally not executed and displays no results.**  It documents the
code that produced the de-identified datasets in `data/prepared/`.  The reviewer-facing main
notebook (`kd_ivig_analysis.ipynb`) starts directly from those files.

Stages covered here:

1. unit / decimal repair, physiological screening, imputation, and the fixed train/validation split;
2. Tabula synthetic-pool training and sampling (the only long-running step);
3. SPA selection with the confidence + PCS dual threshold;
4. export of the public, de-identified files.

Run this notebook only if you need to rebuild the datasets from the raw 2025 snapshots.


## 0. Setup


In [ ]:
from pathlib import Path
import json
import shutil
import subprocess
import sys

import numpy as np
import pandas as pd

ROOT = Path.cwd()
RAW = ROOT / "data" / "raw"
PROCESSED = ROOT / "data" / "processed"
RETAINED = ROOT / "data" / "retained"
PREPARED = ROOT / "data" / "prepared"
sys.path.insert(0, str(ROOT / "tools"))


## 1. Cleaning, imputation, and split

`tools/retained_data.py` implements the complete cleaning pipeline:

* unit / decimal repair for the retained 2025 snapshots;
* physiological screen (impossible values) and training-median imputation for
  Original / SMOTE / validation;
* fixed train/validation split (the validation partition is never resampled);
* Syn / SPA: drop only records with a physiologically impossible value, keep plausible
  values outside the training min/max, and keep fractional day variables;
* derived composites for Panel B.

The cell below is the entry point; it is left uncommented only as code.


In [ ]:
# `tools/retained_data.py` implements this stage.  It reads the raw 2025 snapshots and the
# trained model checkpoints, neither of which is redistributed with this repository, so the
# call is documented as code and left commented out.
#
# sys.argv = ["retained_data.py", "--out", str(RETAINED)]
# retained_data.main()


## 2. Tabula synthetic-pool training

`tools/tabula_synthesis.py` trains Tabula on the SMOTE-balanced training table and samples
the candidate synthetic pool.  This is the only step that takes hours on one GPU; it is not
run here.  The retained output used in the paper is the 2025 synthetic pool; the v3 cleaning
step above then removes only records with a physiologically impossible value.


In [ ]:
# Long-running; run from the repository root when rebuilding from scratch.
# cmd = [
#     sys.executable, "tools/tabula_synthesis.py",
#     "--stage", "all",
#     "--n-samples", "10000",
#     "--out", str(PROCESSED / "synthetic_pool.csv"),
# ]
# subprocess.run(cmd, check=True)


## 3. SPA selection (confidence + PCS)

The 2025 SPA rule: fit a quality-control classifier on SMOTE, score each synthetic record,
perturb it with Gaussian noise, and retain records with

* confidence: `|p - 0.5| > tau_s`;
* perturbation change sensitivity: `PCS = |p(x) - p(x + noise)| >= tau_pcs`.

The 2025 operating point is `tau_s = 0.20`, `tau_pcs = 0.20`; the sensitivity analysis in the
main notebook varies both.  The function below documents the rule; the retained v3 SPA file
is the 2025 `sds` pool after the cleaning step above.


In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import MinMaxScaler


def select_spa(syn: pd.DataFrame, smote: pd.DataFrame, features: list[str],
               tau_s: float = 0.20, tau_pcs: float = 0.20,
               sigma: float = 0.10, seed: int = 42) -> pd.DataFrame:
    """Return the synthetic records passing the confidence + PCS screen."""
    rng = np.random.default_rng(seed)
    Xq = smote[features].astype(float).to_numpy()
    yq = smote["Label"].astype(int).to_numpy()
    scaler = MinMaxScaler().fit(Xq)
    qc = RandomForestClassifier(n_estimators=300, random_state=seed, n_jobs=-1).fit(
        scaler.transform(Xq), yq)

    Xs = scaler.transform(syn[features].astype(float).to_numpy())
    p = qc.predict_proba(Xs)[:, 1]
    Xp = Xs + rng.normal(0, sigma, size=Xs.shape)
    pcs = np.abs(p - qc.predict_proba(Xp)[:, 1])
    keep = (np.abs(p - 0.5) > tau_s) & (pcs >= tau_pcs)
    return syn.loc[keep].copy()


## 4. Export the public prepared files

The main notebook reads only `data/prepared/`.  Files are renamed to make the role of each
split explicit; `row_id` is a sequential row number and no patient identifiers are present.


In [ ]:
PREPARED.mkdir(parents=True, exist_ok=True)
stems = {
    "ori": "original_train",
    "smote": "smote_train",
    "syn": "synthetic_train",
    "spa": "spa_train",
    "val": "validation",
}
for old, new in stems.items():
    for panel in ("a", "b"):
        shutil.copy2(RETAINED / f"{old}_panel_{panel}.csv",
                     PREPARED / f"{new}_panel_{panel}.csv")

# The manifest and README in data/prepared are generated alongside the files.


## 5. What the reviewer needs

The main notebook does **not** import this notebook or rerun these stages.  It reads the
de-identified CSV files in `data/prepared/` and reproduces Figure 2, the SPA sensitivity
heatmap, and the subsequent analyses.
